# 1 — PyTorch baseline
Runs in the **Jupyter container on node-serve-model**.

In [ ]:
import time
import numpy as np
import torch

In [ ]:
import sys
sys.path.insert(0, "models")
model = torch.load("model_artifacts/smartqueue_ranker.pt", map_location="cpu", weights_only=False)
model.eval()
print(model)

In [ ]:
def benchmark_pytorch(model, input_dim=64, num_trials=500, batch_size=32):
    single = torch.randn(1, input_dim)
    with torch.no_grad():
        for _ in range(20):
            model(single)
    latencies = []
    with torch.no_grad():
        for _ in range(num_trials):
            t0 = time.time()
            model(single)
            latencies.append(time.time() - t0)
    print(f"p50: {np.percentile(latencies, 50)*1000:.2f} ms")
    print(f"p95: {np.percentile(latencies, 95)*1000:.2f} ms")
    print(f"p99: {np.percentile(latencies, 99)*1000:.2f} ms")
    batch = torch.randn(batch_size, input_dim)
    times = []
    with torch.no_grad():
        for _ in range(200):
            t0 = time.time()
            model(batch)
            times.append(time.time() - t0)
    print(f"Throughput (batch={batch_size}): {batch_size * 200 / sum(times):.1f} samples/sec")

In [ ]:
print("=== Baseline PyTorch (CPU) ===")
benchmark_pytorch(model)